In [1]:
import requests
from bs4 import BeautifulSoup
import re
from bs4.element import Tag, NavigableString
import pandas as pd

In [2]:
url='https://www.court.gov.mo/zh/subpage/sell'
headers={'USER-AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36 Edg/128.0.0.0'}
r=requests.get(url, headers=headers)
soup=BeautifulSoup(r.content, 'html.parser')

In [3]:
def get_tag_data(soup_tag, result=None):
    if result is None:
        result = {}
    
    # Skip if the tag is h3, h4, ul, or li, but process its children
    if isinstance(soup_tag, Tag) and soup_tag.name in ('h3', 'h4', 'ul', 'li'):
        for child in soup_tag.children:
            result = get_tag_data(child, result)
        return result
    
    # If the tag is a div, process its contents for <strong> and text
    if isinstance(soup_tag, Tag) and soup_tag.name == 'div':
        current_key = None
        current_value = ''
        for item in soup_tag.contents:
            if isinstance(item, Tag) and item.name == 'strong' and item.get('style') != 'visibility: hidden':
                # Save previous key-value pair if exists
                if current_key and current_value.strip():
                    if current_key in result:
                        if isinstance(result[current_key], list):
                            result[current_key].append(current_value.strip())
                        else:
                            result[current_key] = [result[current_key], current_value.strip()]
                    else:
                        result[current_key] = current_value.strip()
                # Set new key
                current_key = re.sub(r'[:：-]$', '', item.get_text().strip())
                current_value = ''
            elif isinstance(item, NavigableString):
                # Append text to current value
                current_value += item.strip()
            elif isinstance(item, Tag) and item.name == 'br':
                # Save on <br/> if key and value exist
                if current_key and current_value.strip():
                    if current_key in result:
                        if isinstance(result[current_key], list):
                            result[current_key].append(current_value.strip())
                        else:
                            result[current_key] = [result[current_key], current_value.strip()]
                    else:
                        result[current_key] = current_value.strip()
                current_value = ''
        
        # Save final key-value pair in this div
        if current_key and current_value.strip():
            if current_key in result:
                if isinstance(result[current_key], list):
                    result[current_key].append(current_value.strip())
                else:
                    result[current_key] = [result[current_key], current_value.strip()]
            else:
                result[current_key] = current_value.strip()
    
    # Recursively process child tags
    if isinstance(soup_tag, Tag):
        for child in soup_tag.children:
            result = get_tag_data(child, result)
    
    return result

In [4]:
sell_item=soup.find_all('div', {'class': 'sellitem'})
item_list=[]
for item in sell_item:
    date=item.find('div', {'class': 'selldate'}).text.strip()
    item_info=item.find_all('div', {'class': 'sellinfo'})
    case_info=None
    sell_info=None
    sell_property=None
    for info in item_info:
        if '案件資料' in info.text:
            case_info=get_tag_data(info)
        elif '變賣資料' in info.text:
            sell_info=get_tag_data(info)
        elif '變賣財產' in info.text:
            sell_property=get_tag_data(info)
    # item_list.append({
    #     'date': date,
    #     'case_info': case_info,
    #     'sell_info': sell_info,
    #     'sell_property': sell_property
    # })
    result={'date': date, **case_info, **sell_info, **sell_property}
    item_list.append(result)

In [5]:
df = pd.DataFrame(item_list)

# Ensure all columns contain hashable types by converting lists to strings
df = df.applymap(lambda x: str(x) if isinstance(x, list) else x)

# Load existing records
record = pd.read_csv('court_sell.csv')

# Concatenate the new and existing records
df = pd.concat([df, record], ignore_index=True)

# Drop duplicates
df = df.drop_duplicates()

# Save the updated dataframe to the CSV file
df.to_csv('court_sell.csv', index=False)



C:\Users\tonyfong\AppData\Local\Temp\ipykernel_6932\1732576972.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: str(x) if isinstance(x, list) else x)
